# Solution: Transforming Time-Series Data in pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
BIKE_PATH = "../data/bikeshare_rides.csv"
TEMP_PATH = "../data/melbourne_temps.csv"

Before fitting ARIMA, you need to know which series is already stationary and which needs transformation. Complete the analysis for both datasets.

In [ ]:
# Load both datasets with DatetimeIndex
bike = pd.read_csv(BIKE_PATH, parse_dates=["date"]).set_index("date")["rides"].sort_index()
temp = pd.read_csv(TEMP_PATH, parse_dates=["Date"]).set_index("Date")["Temp"].sort_index()

In [ ]:
# Compute 30-day rolling statistics for the bike rides
bike_mean = bike.rolling(30).mean()
bike_std = bike.rolling(30).std()

# Run ADF test on raw bike rides
bike_adf = adfuller(bike.dropna(), autlag="AIC")
print(f"Bike ADF p-value: {bike_adf[1]:.4f}")

In [ ]:
# Apply first differencing to bike rides if non-stationary
bike_diff = bike.diff().dropna()
bike_adf2 = adfuller(bike_diff, autlag="AIC")
print(f"Bike differenced p-value: {bike_adf2[1]:.4f}")

In [ ]:
# Repeat for Melbourne temperatures
temp_mean = temp.rolling(30).mean()
temp_std = temp.rolling(30).std()
temp_adf = adfuller(temp.dropna(), autlag="AIC")
print(f"Temp ADF p-value: {temp_adf[1]:.4f}")

In [ ]:
# Build comparison table
report = pd.DataFrame({
    "Bike Rides": [bike_adf[1], bike_adf2[1]],
    "Melbourne Temps": [temp_adf[1], temp_adf[1]]
}, index=["Before", "After"])
print(report)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(bike.index, bike.values, color=UB["Brand Blue"], label="Bike Rides")
axes[0].plot(bike_mean.index, bike_mean.values, color=US["Orange"], label="30-day mean")
axes[0].fill_between(bike_std.index, bike_mean.values - bike_std.values, bike_mean.values + bike_std.values, color=US["Orange"], alpha=0.2)
axes[0].set_title("Bike Rides Rolling Stats", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Rides", fontsize=16)
axes[0].tick_params(axis="both", labelsize=14)
axes[0].legend()
axes[1].plot(bike_diff.index, bike_diff.values, color=UB["Medium Blue"])
axes[1].axhline(0, color=UN["Dark Gray"], linestyle="--")
axes[1].set_title("Bike Rides Differenced", fontsize=18, fontweight="bold")
axes[1].set_ylabel("Change in Rides", fontsize=16)
axes[1].tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()